In [1]:
import os, glob, cv2, numpy as np, pandas as pd, torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

ROOT = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
print(ROOT, os.listdir(ROOT))

tr, va, te = [pd.read_csv(f"{ROOT}/{n}.csv") for n in ("train", "val", "test")]
print(len(tr), len(va), len(te))

/kaggle/input/datasets/abhisheksharma111777/rsna-png256-splits ['boxes.csv', 'val.csv', 'train.csv', 'test.csv', 'images_256.npy']
18678 4003 4003


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

f = lambda d: (d["view"] == "AP").astype(int).values.reshape(-1, 1)
clf = LogisticRegression(class_weight="balanced").fit(f(tr), tr["label"])
pred = clf.predict(f(va))
print(classification_report(va["label"], pred,     target_names=["Normal", "Not Normal", "Lung Opacity"], digits=3))

              precision    recall  f1-score   support

      Normal      0.501     0.806     0.618      1327
  Not Normal      0.000     0.000     0.000      1774
Lung Opacity      0.379     0.785     0.511       902

    accuracy                          0.444      4003
   macro avg      0.293     0.530     0.376      4003
weighted avg      0.252     0.444     0.320      4003



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
IMGS = np.load(f"{ROOT}/images_256.npy")
print("Image array:", IMGS.shape)

class CXR(Dataset):
    def __init__(self, df, train=False):
        self.idx, self.y, self.train = df["idx"].values, df["label"].values, train
        # small rotation/crop/brightness only; no flips (heart position matters)
        self.aug = T.Compose([T.RandomRotation(7),
                              T.RandomResizedCrop(256, scale=(0.9, 1.0), antialias=True),
                              T.ColorJitter(0.15, 0.15)])
        self.norm = T.Normalize(MEAN, STD)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        img = np.array(IMGS[self.idx[i]])
        x = torch.from_numpy(img).float().div(255).unsqueeze(0).repeat(3, 1, 1)
        if self.train: x = self.aug(x)
        return self.norm(x), int(self.y[i])

train_dl = DataLoader(CXR(tr, True), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(CXR(va), batch_size=64, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_dl))
print(xb.shape, yb[:16])

Image array: (26684, 256, 256)
torch.Size([64, 3, 256, 256]) tensor([2, 0, 1, 2, 1, 0, 0, 1, 1, 2, 1, 1, 2, 0, 0, 1])


In [6]:
import torch.nn as nn
from sklearn.metrics import classification_report, roc_auc_score
import time
from tqdm.auto import tqdm
from sklearn.metrics import recall_score

device = "cuda"
scaler = torch.amp.GradScaler("cuda")

class SmallCNN(nn.Module):
    def __init__(self, n=3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Linear(256, n)
    def forward(self, x):
        return self.head(self.f(x).flatten(1))


def run_epoch(model, dl, opt=None, loss_fn=None, desc=""):
    training = opt is not None
    model.train(training)
    losses, probs, ys = [], [], []
    bar = tqdm(dl, desc=desc, leave=False)
    with torch.set_grad_enabled(training):
        for x, y in bar:
            x, y = x.to(device, non_blocking=True), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(x)
                loss = loss_fn(out, y)
            if training:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(loss.item())
            if not training:                      # predictions only needed for validation
                probs.append(out.detach().float().softmax(1).cpu())
                ys.append(y.detach().cpu())
            bar.set_postfix(loss=f"{np.mean(losses):.3f}")
    if training:
        return np.mean(losses), None, None
    return np.mean(losses), torch.cat(probs).numpy(), torch.cat(ys).numpy()

def fit(model, epochs, lr, name):
    model.to(device)
    w = torch.tensor(len(tr) / (3 * np.bincount(tr["label"])), dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=w)          # class weights for imbalance
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best = 0
    print(f"{'ep':>3} {'train_loss':>10} {'val_loss':>8} {'val_acc':>7} {'AUROC':>6} | recall: Normal / NotNormal / Opacity | time")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        trl, _, _ = run_epoch(model, train_dl, opt, loss_fn, desc=f"epoch {ep}/{epochs} train")
        vl, p, y = run_epoch(model, val_dl, None, loss_fn, desc=f"epoch {ep}/{epochs} val")
        pred = p.argmax(1)
        acc = (pred == y).mean()
        auc = roc_auc_score(y, p, multi_class="ovr", average="macro")
        rec = recall_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
        mark = ""
        if auc > best:
            best = auc
            torch.save(model.state_dict(), f"/kaggle/working/{name}.pt")
            mark = "  <- best, saved"
        print(f"{ep:>3} {trl:>10.3f} {vl:>8.3f} {acc:>7.3f} {auc:>6.3f} | "
              f"{rec[0]:.2f} / {rec[1]:.2f} / {rec[2]:.2f} | {time.time()-t0:>4.0f}s{mark}", flush=True)
    model.load_state_dict(torch.load(f"/kaggle/working/{name}.pt"))
    return model

## Baseline results (validation set)
| Model | Accuracy | Macro F1 | Macro AUROC | Lung Opacity recall (AP / PA) |
|---|---|---|---|---|
| View-only (AP/PA) | 0.444 | 0.376 | - | - |
| Small CNN, 8 epochs | 0.634 | 0.620 | 0.828 | 0.638 / 0.144 |
Finding: the small CNN rarely detects pneumonia in PA images, so the AP/PA shortcut is still a factor.

In [8]:
import json , random ,time
from pathlib import Path
import torch.nn as nn,torchvision
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import (classification_report,roc_auc_score,f1_score,recall_score,confusion_matrix)

device = "cuda"
SEED = 42
CLASS_NAMES = ["Normal", "Not Normal", "Lung Opacity"]
RES = Path("/kaggle/working/results"); RES.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)



In [9]:
RESULTS = {"small_cnn": {"accuracy": 0.634, "macro_f1": 0.620, "macro_auroc": 0.828,
                         "recall": {"Normal": 0.937, "Not Normal": 0.460, "Lung Opacity": 0.532},
                         "AP_macro_f1": 0.594, "PA_macro_f1": 0.512,
                         "AP_opacity_recall": 0.638, "PA_opacity_recall": 0.144}}

In [10]:
@torch.no_grad()
def predict(model,dl):
    model.eval()
    probs, ys = [],[]
    for x, y in dl:
        with torch.autocast("cuda",dtype = torch.float16):
            out = model(x.to(device,non_blocking = True))

        probs.append(out.float().softmax(1).cpu());ys.append(y)

    return torch.cat(probs).numpy(),torch.cat(ys).numpy()
    
        

In [11]:
def compute_metrics(p,y,views):
    pred = p.argmax(1)
    rec = recall_score(y,pred,labels = [0,1,2],average = None,zero_division= 0)
    m = {"accuracy": float((pred == y).mean()),
         "macro_f1": float(f1_score(y, pred, average="macro")),
         "macro_auroc": float(roc_auc_score(y, p, multi_class="ovr", average="macro")),
         "recall": {n: round(float(r), 4) for n, r in zip(CLASS_NAMES, rec)}}
    for v in ["AP", "PA"]:
        k = views == v
        m[f"{v}_macro_f1"] = float(f1_score(y[k], pred[k], average="macro"))
        m[f"{v}_opacity_recall"] = float(recall_score(y[k], pred[k], labels=[2], average=None, zero_division=0)[0])
    return m

In [12]:
def plot_curves(hist, name):
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(hist["val_auroc"], label="val macro-AUROC"); ax[1].plot(hist["val_f1"], label="val macro-F1")
    ax[1].set_title("Validation metrics"); ax[1].legend()
    for a in ax: a.set_xlabel("epoch")
    fig.tight_layout(); fig.savefig(RES / f"{name}_curves.png", dpi=150); plt.show()

In [13]:
def plot_confusion(p, y, name):
    cm = confusion_matrix(y, p.argmax(1), normalize="true")
    fig, ax = plt.subplots(figsize=(4.8, 4))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(name)
    fig.tight_layout(); fig.savefig(RES / f"{name}_confusion.png", dpi=150); plt.show()

In [14]:
def comparison_table():
    rows = [{"model": n, "macro_F1": m["macro_f1"], "macro_AUROC": m["macro_auroc"],
             "Opacity_recall": m["recall"]["Lung Opacity"],
             "AP_macro_F1": m["AP_macro_f1"], "PA_macro_F1": m["PA_macro_f1"],
             "PA_Opacity_recall": m["PA_opacity_recall"]} for n, m in RESULTS.items()]
    df = pd.DataFrame(rows).set_index("model").round(3)
    df.to_csv(RES / "comparison.csv")
    return df

DenseNet 

In [15]:
def make_densenet(n=3,dropout= 0.2):
    m = torchvision.models.densenet121(weights = "DEFAULT")
    m.classifier = nn.Sequential(nn.Dropout(dropout),nn.Linear(m.classifier.in_features,n))
    return m
    
def make_loaders(balance_views, bs=64, workers=4):
    if balance_views:
        g = tr["label"].astype(str) + "_" + tr["view"]
        w = 1.0 / g.map(g.value_counts()).values
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                        num_samples=len(tr), replacement=True)
        tdl = DataLoader(CXR(tr, True), batch_size=bs, sampler=sampler,
                         num_workers=workers, pin_memory=True, drop_last=True)
    else:
        tdl = DataLoader(CXR(tr, True), batch_size=bs, shuffle=True,
                         num_workers=workers, pin_memory=True, drop_last=True)
    vdl = DataLoader(CXR(va), batch_size=bs, num_workers=workers, pin_memory=True)
    return tdl, vdl

def train_experiment(name, make_model, epochs=10, lr=3e-4, balance_views=False, patience=4):
    set_seed()
    tdl, vdl = make_loaders(balance_views)
    model = make_model().to(device)

    if balance_views:
        loss_fn = nn.CrossEntropyLoss()                # the sampler already balances classes
    else:
        w = torch.tensor(len(tr) / (3 * np.bincount(tr["label"])), dtype=torch.float32, device=device)
        loss_fn = nn.CrossEntropyLoss(weight=w)        # class weights handle the imbalance

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tdl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda")

    hist = {k: [] for k in ["train_loss", "val_loss", "val_auroc", "val_f1"]}
    best, bad, ckpt = -1.0, 0, f"/kaggle/working/{name}.pt"
    print(f"{'ep':>3} {'train':>6} {'val':>6} {'AUROC':>6} {'F1':>6} | recall Normal/NotNormal/Opacity | time")

    for ep in range(1, epochs + 1):
        t0 = time.time(); model.train(); tl = []
        for x, y in tqdm(tdl, desc=f"epoch {ep}/{epochs}", leave=False, mininterval=30):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl.append(loss.item())

        p, yv = predict(model, vdl)
        vl = float(-np.log(p[np.arange(len(yv)), yv] + 1e-8).mean())   # unweighted cross-entropy
        pred = p.argmax(1)
        auc = roc_auc_score(yv, p, multi_class="ovr", average="macro")
        f1 = f1_score(yv, pred, average="macro")
        rec = recall_score(yv, pred, labels=[0, 1, 2], average=None, zero_division=0)
        for k, v in zip(hist, [np.mean(tl), vl, auc, f1]): hist[k].append(float(v))

        mark = ""
        if auc > best:
            best, bad = auc, 0
            torch.save(model.state_dict(), ckpt); mark = " <- best"
        else:
            bad += 1
        print(f"{ep:>3} {np.mean(tl):>6.3f} {vl:>6.3f} {auc:>6.3f} {f1:>6.3f} | "
              f"{rec[0]:.2f} / {rec[1]:.2f} / {rec[2]:.2f} | {time.time()-t0:>4.0f}s{mark}", flush=True)
        if bad >= patience:
            print(f"Early stop: no AUROC improvement for {patience} epochs"); break

    model.load_state_dict(torch.load(ckpt))              # go back to the best epoch
    p, yv = predict(model, vdl)
    m = compute_metrics(p, yv, va["view"].values)
    m.update(epochs_run=len(hist["val_auroc"]), lr=lr, balance_views=balance_views, seed=SEED)
    RESULTS[name] = m
    json.dump(m, open(RES / f"{name}_metrics.json", "w"), indent=2)
    json.dump(hist, open(RES / f"{name}_history.json", "w"))
    np.save(RES / f"{name}_val_probs.npy", p)
    plot_curves(hist, name); plot_confusion(p, yv, name)
    print(classification_report(yv, p.argmax(1), labels=[0, 1, 2], target_names=CLASS_NAMES, digits=3, zero_division=0))
    return model, p

In [ ]:
# _ = train_experiment("smoke_test", make_densenet, epochs=1, lr=3e-4, balance_views=True)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 199MB/s]


 ep  train    val  AUROC     F1 | recall Normal/NotNormal/Opacity | time


  1  0.729  0.694  0.860  0.680 | 0.89 / 0.52 / 0.71 |  126s <- best


In [ ]:
dn_plain, p_plain = train_experiment("densenet121_plain", make_densenet, epochs=10, lr=3e-4)
try:
    dn_bal, p_bal = train_experiment("densenet121_viewbalanced", make_densenet,
                                       epochs=10, lr=3e-4, balance_views=True)
except Exception as e:
    print("View-balanced run failed:", repr(e))

comparison_table()